# セクターローテーション分析
過去5年の週次データで各セクターのベスト3・ワースト3を集計します。

In [ ]:
!pip install yfinance pandas -q

In [ ]:
import yfinance as yf
import pandas as pd

# TOPIX-17 セクターETF（日興AM 1615〜1631）
SECTOR_ETFS = {
    '1615.T': '食品',
    '1616.T': 'エネルギー資源',
    '1617.T': '建設・資材',
    '1618.T': '素材・化学',
    '1619.T': '医薬品',
    '1620.T': '自動車・輸送機',
    '1621.T': '鉄鋼・非鉄',
    '1622.T': '機械',
    '1623.T': '電機・精密',
    '1624.T': '情報通信・サービス',
    '1625.T': '電力・ガス',
    '1626.T': '不動産',
    '1627.T': '金融（除く銀行）',
    '1628.T': '銀行業',
    '1629.T': '運輸・物流',
    '1630.T': '商社・卸売',
    '1631.T': '小売業',
}
print(f'対象ETF: {len(SECTOR_ETFS)}本')

In [ ]:
# データ取得（数秒で完了）
tickers = list(SECTOR_ETFS.keys())

raw = yf.download(
    tickers,
    period='5y',
    interval='1wk',
    auto_adjust=True,
    progress=True,
)

close = raw['Close'].rename(columns=SECTOR_ETFS)
close = close.dropna(how='all')
print(f'期間: {close.index[0].date()} 〜 {close.index[-1].date()}')
print(f'週数: {len(close)}')
close.tail(3)

In [ ]:
# 週次リターン計算
weekly_ret = close.pct_change().dropna(how='all')
weekly_ret.tail(3)

In [ ]:
# 週次ベスト3・ワースト3 一覧
records = []
for date, row in weekly_ret.iterrows():
    row_clean = row.dropna()
    if len(row_clean) < 5:
        continue
    ranked = row_clean.sort_values(ascending=False)
    records.append({
        '週':        date.strftime('%Y-%m-%d'),
        '1位':       ranked.index[0],
        '1位_%':     f'{ranked.iloc[0]*100:+.1f}%',
        '2位':       ranked.index[1],
        '2位_%':     f'{ranked.iloc[1]*100:+.1f}%',
        '3位':       ranked.index[2],
        '3位_%':     f'{ranked.iloc[2]*100:+.1f}%',
        '最下位':    ranked.index[-1],
        '最下位_%':  f'{ranked.iloc[-1]*100:+.1f}%',
        '最下位2':   ranked.index[-2],
        '最下位2_%': f'{ranked.iloc[-2]*100:+.1f}%',
        '最下位3':   ranked.index[-3],
        '最下位3_%': f'{ranked.iloc[-3]*100:+.1f}%',
    })

df_result = pd.DataFrame(records).set_index('週')
df_result

In [ ]:
# 総合集計（セクター別の強さ・弱さ傾向）
total_weeks = len(weekly_ret.dropna())

summary = pd.DataFrame({
    '平均リターン(%)': (weekly_ret.mean() * 100).round(2),
    '勝率(%)':        (weekly_ret > 0).mean().mul(100).round(1),
    '首位回数':        weekly_ret.apply(lambda r: r.dropna().idxmax() if r.dropna().any() else None, axis=1).value_counts(),
    '最下位回数':      weekly_ret.apply(lambda r: r.dropna().idxmin() if r.dropna().any() else None, axis=1).value_counts(),
}).fillna(0)

summary['首位率(%)']   = (summary['首位回数']   / total_weeks * 100).round(1)
summary['最下位率(%)'] = (summary['最下位回数'] / total_weeks * 100).round(1)
summary = summary.sort_values('平均リターン(%)', ascending=False)
print(f'集計週数: {total_weeks}週')
summary

In [ ]:
# 月別・セクター平均リターン（ヒートマップ用）
monthly = weekly_ret.copy()
monthly.index = pd.to_datetime(monthly.index)
monthly['month'] = monthly.index.month

month_labels = {1:'1月',2:'2月',3:'3月',4:'4月',5:'5月',6:'6月',
                7:'7月',8:'8月',9:'9月',10:'10月',11:'11月',12:'12月'}

df_monthly = monthly.groupby('month')[list(SECTOR_ETFS.values())].mean() * 100
df_monthly.index = df_monthly.index.map(month_labels)
df_monthly.round(2)

In [ ]:
# CSV保存
from google.colab import files

df_result.to_csv('sector_rotation_weekly.csv', encoding='utf-8-sig')
summary.to_csv('sector_summary.csv', encoding='utf-8-sig')
df_monthly.to_csv('sector_monthly.csv', encoding='utf-8-sig')

files.download('sector_rotation_weekly.csv')
files.download('sector_summary.csv')
files.download('sector_monthly.csv')